# Cosine Similarity Distribution for Headline Pairs

This notebook uploads a dataset like `nonsarcastic_to_sarcastic_out.json` at runtime, computes cosine similarity between `headline` and `reversed_headline` for each sample using the existing implementation in `evaluation/text_similarity.py`, and plots the similarity distribution.


In [ ]:
# Pull code, install dependencies

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LINCHENYU2030S/CS4248_group_project.git"
REPO_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_EVAL_DIR = LOCAL_PROJECT_ROOT / "evaluation"
LOCAL_EVAL_DIR = Path("/content/evaluation")

current_dir = Path.cwd().resolve()
if current_dir.name == "evaluation":
    project_eval_dir = current_dir
elif current_dir.name == "notebooks" and (current_dir.parent / "evaluation").exists():
    project_eval_dir = current_dir.parent / "evaluation"
elif (current_dir / "evaluation").exists():
    project_eval_dir = current_dir / "evaluation"
elif LOCAL_PROJECT_EVAL_DIR.exists():
    project_eval_dir = LOCAL_PROJECT_EVAL_DIR
elif LOCAL_EVAL_DIR.exists():
    project_eval_dir = LOCAL_EVAL_DIR
elif "google.colab" in sys.modules:
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    project_eval_dir = REPO_ROOT / "evaluation"
else:
    raise RuntimeError(
        f"Could not locate the repo evaluation directory from: {current_dir}"
    )

os.chdir(project_eval_dir)
PROJECT_EVAL_DIR = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_EVAL_DIR.parent
if PROJECT_EVAL_DIR.name != "evaluation":
    raise RuntimeError(f"Expected to run inside the evaluation directory, got: {PROJECT_EVAL_DIR}")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
for path in [PROJECT_ROOT, PROJECT_EVAL_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "sentence-transformers>=2.6.0",
        "transformers>=4.39.0",
        "pandas>=2.0.0",
        "matplotlib>=3.7.0",
        "ipywidgets>=8.1.0",
        "tqdm>=4.66.0",
    ],
    check=True,
)

print(f"Working directory: {PROJECT_EVAL_DIR}")
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# Upload dataset at run time

import shutil

DATASET_FILENAME = "nonsarcastic_to_sarcastic_out.json"
runtime_dataset_path = PROJECT_EVAL_DIR / DATASET_FILENAME

if runtime_dataset_path.exists():
    print(f"Dataset already available at: {runtime_dataset_path}")
elif "google.colab" in sys.modules:
    from google.colab import files

    print(f"Upload {DATASET_FILENAME} when the file picker opens.")
    uploaded = files.upload()

    if DATASET_FILENAME not in uploaded:
        raise FileNotFoundError(
            f"Expected {DATASET_FILENAME!r}. Uploaded files: {list(uploaded)}"
        )

    upload_candidates = [Path("/") / DATASET_FILENAME, Path.cwd() / DATASET_FILENAME]
    upload_source = next((path for path in upload_candidates if path.exists()), None)
    if upload_source is None:
        raise FileNotFoundError(
            f"Uploaded file not found in expected locations: {upload_candidates}"
        )

    shutil.move(str(upload_source), str(runtime_dataset_path))
    print(f"Dataset saved to: {runtime_dataset_path}")
else:
    raise FileNotFoundError(
        f"Dataset not found at {runtime_dataset_path}. Place the JSON file there before continuing."
    )

DATASET_PATH = runtime_dataset_path


In [ ]:
# Imports and configuration

import json

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

from evaluation.text_similarity import (
    DEFAULT_MODEL_NAME,
    batch_cosine_similarity,
    load_embedding_model,
)

pd.set_option("display.max_colwidth", 120)

HEADLINE_COLUMN = "headline"
REVERSED_HEADLINE_COLUMN = "reversed_headline"
COSINE_COLUMN = "cosine_similarity"
EMBEDDING_MODEL_NAME = DEFAULT_MODEL_NAME
BATCH_SIZE = 256
OUTPUT_PATH = PROJECT_EVAL_DIR / (Path(DATASET_FILENAME).stem + "_with_cosine_similarity.json")

print(f"Dataset path: {DATASET_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Output path: {OUTPUT_PATH}")


In [ ]:
# Load dataset

try:
    df = pd.read_json(DATASET_PATH, lines=True)
except ValueError:
    df = pd.read_json(DATASET_PATH)

required_columns = {HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=[HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN]).copy()
df[HEADLINE_COLUMN] = df[HEADLINE_COLUMN].astype(str).str.strip()
df[REVERSED_HEADLINE_COLUMN] = df[REVERSED_HEADLINE_COLUMN].astype(str).str.strip()
df = df[(df[HEADLINE_COLUMN] != "") & (df[REVERSED_HEADLINE_COLUMN] != "")].copy()

display(df[[HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN]].head())
print(f"Rows to score: {len(df):,}")


In [ ]:
# Compute cosine similarity in batches using evaluation/text_similarity.py

print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
_ = load_embedding_model(EMBEDDING_MODEL_NAME)

headlines = df[HEADLINE_COLUMN].tolist()
reversed_headlines = df[REVERSED_HEADLINE_COLUMN].tolist()
all_similarities = []

for start in tqdm(range(0, len(df), BATCH_SIZE), desc="Computing cosine similarity"):
    end = start + BATCH_SIZE
    batch_scores = batch_cosine_similarity(
        headlines[start:end],
        reversed_headlines[start:end],
        model_name=EMBEDDING_MODEL_NAME,
    )
    all_similarities.extend(batch_scores)

df[COSINE_COLUMN] = all_similarities
display(df[[HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN, COSINE_COLUMN]].head())


In [ ]:
# Summary statistics and histogram

summary = df[COSINE_COLUMN].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame(name=COSINE_COLUMN)
display(summary)

plt.figure(figsize=(10, 6))
plt.hist(df[COSINE_COLUMN], bins=40, edgecolor="black")
plt.xlabel("Cosine similarity")
plt.ylabel("Frequency")
plt.title("Cosine similarity distribution: headline vs reversed_headline")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# Filter rows with cosine similarity >= 0.4 and save only the original dataset fields

FILTER_THRESHOLD = 0.4
FILTERED_OUTPUT_PATH = PROJECT_EVAL_DIR / "nonsarcastic_to_sarcastic_filtered.json"
original_columns = [column for column in df.columns if column != COSINE_COLUMN]

filtered_df = df[df[COSINE_COLUMN] >= FILTER_THRESHOLD].copy()

with FILTERED_OUTPUT_PATH.open("w", encoding="utf-8") as fout:
    for record in filtered_df[original_columns].to_dict(orient="records"):
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Kept {len(filtered_df):,} / {len(df):,} rows with cosine similarity >= {FILTER_THRESHOLD}")
print(f"Saved filtered dataset to: {FILTERED_OUTPUT_PATH}")
display(filtered_df[original_columns + [COSINE_COLUMN]].head(10))


In [ ]:
# Optional: save the scored dataset as JSONL and inspect the lowest-similarity pairs

with OUTPUT_PATH.open("w", encoding="utf-8") as fout:
    for record in df.to_dict(orient="records"):
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved scored dataset to: {OUTPUT_PATH}")
display(df.nsmallest(10, COSINE_COLUMN)[[HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN, COSINE_COLUMN]])
